# Resurfacing features

This notebook cleans the DOT resurfacing file and builds the street-season resurfacing table. Output: `data/derived/resurfacing_agg_full.csv`.


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/raw/roadwork/dot_inhouse_resurfacing.csv")

/var/folders/zh/4snkqr657fb06hs2r74_4xpm0000gn/T/ipykernel_1705/2576664700.py:1: DtypeWarning: Columns (5,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/raw/roadwork/dot_inhouse_resurfacing.csv")


In [3]:
# Column names
milling_start = "Location Actual Milling Start Date"
milling_end   = "Location Actual Milling End Date"
paving_start  = "Location Actual Paving Start Date"
paving_end    = "Location Actual Paving End Date"

# Create boolean masks for non-missing pairs
milling_mask = df[milling_start].notna() & df[milling_end].notna()
paving_mask  = df[paving_start].notna() & df[paving_end].notna()

# Keep rows where either condition is True
resurfacing = df[milling_mask | paving_mask].copy()

In [4]:
resurfacing.drop(columns = ['Project Speed Bumps'], inplace = True)

In [5]:
# remove time portion by normalizing
resurfacing[milling_start] = pd.to_datetime(resurfacing[milling_start]).dt.normalize()
resurfacing[milling_end]   = pd.to_datetime(resurfacing[milling_end]).dt.normalize()
resurfacing[paving_start]  = pd.to_datetime(resurfacing[paving_start]).dt.normalize()
resurfacing[paving_end]    = pd.to_datetime(resurfacing[paving_end]).dt.normalize()

# compute durations (end - start + 1 day)
resurfacing["milling_time"] = (abs(resurfacing[milling_end] - resurfacing[milling_start])).dt.days.add(1)
resurfacing["paving_time"]  = (abs(resurfacing[paving_end]  - resurfacing[paving_start])).dt.days.add(1)

# total_time treating NaN as 0
resurfacing["total_time"] = (
    resurfacing["milling_time"].fillna(0) +
    resurfacing["paving_time"].fillna(0)
)

# special case: both durations 1 and exact same day
same_day_overlap = (
    (resurfacing["milling_time"] == 1) &
    (resurfacing["paving_time"] == 1) &
    (resurfacing[milling_start] == resurfacing[paving_start]) &
    (resurfacing[milling_end]   == resurfacing[paving_end])
)
resurfacing.loc[same_day_overlap, "total_time"] = 1

In [6]:
# extract year from dates
years_valid = (
    (resurfacing[milling_start].dt.year > 2001) &
    (resurfacing[milling_end].dt.year   > 2001) &
    (resurfacing[paving_start].dt.year  > 2001) &
    (resurfacing[paving_end].dt.year    > 2001)
)

# keep only rows where all four dates are after 2001
resurfacing = resurfacing[years_valid].copy()

IMPUTATION: for a given row with Location Actual Paving Square Yard not populated, impute the value as median across all rows that have that value and that have the same "Project Type" value

In [7]:
from sklearn.impute import SimpleImputer
import pandas as pd

def impute_paving_square_yard(df):
    df = df.copy()
    imputed_groups = []

    for project_type, group in df.groupby("Project Type", dropna=False):
        if pd.notna(project_type):
            imp = SimpleImputer(strategy="median")
            col = "Location Actual Paving Square Yard"

            # Impute only that one column
            group[[col]] = pd.DataFrame(
                imp.fit_transform(group[[col]]),
                index=group.index,
                columns=[col]
            )
        imputed_groups.append(group)

    return pd.concat(imputed_groups, ignore_index=True)

resurfacing = impute_paving_square_yard(resurfacing)


In [8]:
import pandas as pd
import numpy as np

# Columns of interest
cols = [
    "Borough Code",
    "Location On Street",
    "Location Actual Milling Start Date",
    "Location Actual Milling End Date",
    "Location Actual Paving Start Date",
    "Location Actual Paving End Date",
    "Location Actual Paving Square Yard",
]

resurfacing_temp = resurfacing[cols].copy()

# Convert all date columns to datetime (safe coercion)
date_cols = [
    "Location Actual Milling Start Date",
    "Location Actual Milling End Date",
    "Location Actual Paving Start Date",
    "Location Actual Paving End Date",
]
for c in date_cols:
    resurfacing_temp[c] = pd.to_datetime(resurfacing_temp[c], errors="coerce")

# Compute start_date and end_date as min/max across the four date fields (per row)
resurfacing_temp["start_date"] = resurfacing_temp[date_cols].min(axis=1)
resurfacing_temp["end_date"] = resurfacing_temp[date_cols].max(axis=1)

# Keep only desired columns
resurfacing_temp = resurfacing_temp[
    ["Borough Code", "Location On Street", "start_date", "end_date", "Location Actual Paving Square Yard"]
]


In [9]:
#borough code map from DOT to CSCL

borough_map = {'M': '1', 'X': '2', 'B': '3', 'Q': '4', 'S': '5'}  # Manhattan, Bronx, Queens, Brooklyn, Staten Island


In [10]:
from street_normalize import normalize

In [11]:
# Map Borough Code letters to numbers
resurfacing_temp["Borough Number"] = resurfacing_temp["Borough Code"].map(borough_map)

# Apply normalization to street
resurfacing_temp["normalized_street_name"] = (
    resurfacing_temp["Borough Number"].astype(str) + "-" +
    resurfacing_temp["Location On Street"].apply(normalize)
)

# Drop original columns
resurfacing_temp = resurfacing_temp.drop(columns=["Borough Code", "Location On Street", "Borough Number"])

# Rename and reorder columns

resurfacing_temp = resurfacing_temp.rename(columns = {"Location Actual Paving Square Yard": "area"})
resurfacing_temp = resurfacing_temp[
    ["normalized_street_name", "start_date", "end_date", "area"]
]


now aggregate by street, and restrict to PAVING season March 21 - Nov 20 (include days in that range even if full range contains more, truncate); one row for each street

In [12]:
import pandas as pd

# Define roadwork season boundaries (month/day only)
ROADWORK_START = (3, 21)
ROADWORK_END = (11, 20)

def overlaps_roadwork_season(start, end):
    """Return list of years whose roadwork season [Mar 21, Nov 20] overlaps the given interval."""
    if pd.isna(start) or pd.isna(end):
        return []
    if start > end:
        start, end = end, start

    years = []
    for y in range(start.year, end.year + 1):
        s = pd.Timestamp(y, 3, 21)
        e = pd.Timestamp(y, 11, 20, 23, 59, 59)
        if start <= e and end >= s:
            years.append(y)
    return years

records = []
for _, row in resurfacing_temp.iterrows():
    years = overlaps_roadwork_season(row["start_date"], row["end_date"])
    for y in years:
        records.append({
            "normalized_street_name": row["normalized_street_name"],
            "roadwork_season": y,
            "area": row["area"]
        })

resurfacing_agg = (
    pd.DataFrame(records)
    .groupby(["normalized_street_name", "roadwork_season"], as_index=False)[["area"]]
    .sum()
    .round({"area": 1})
    .sort_values(["normalized_street_name", "roadwork_season"], ascending=[True, True])
    .reset_index(drop=True)
)


get total_area from snowfall_df

In [13]:
snowfall = pd.read_csv("data/derived/snowfall_df.csv")

In [14]:
snowfall.head()

,normalized_street_name_season,normalized_street_name,year,total_area,snowfall,snowfall_factor
0,1-1 avenue_2017,1-1 avenue,2017,253446.943078,1.846868e+07,72.87
1,1-1 avenue_2018,1-1 avenue,2018,253446.943078,2.038474e+07,80.43
2,1-1 avenue_2019,1-1 avenue,2019,253446.943078,1.218826e+07,48.09
3,1-1 avenue_2021,1-1 avenue,2021,253446.943078,2.198145e+07,86.73
4,1-1 avenue_2022,1-1 avenue,2022,253446.943078,1.245438e+07,49.14


In [15]:
resurfacing_agg = resurfacing_agg.merge(
    snowfall[['normalized_street_name', 'total_area']],
    on='normalized_street_name',
    how='left'
)

resurfacing_agg['total_area'] = resurfacing_agg['total_area'].round(1)

In [16]:
resurfacing_agg["roadwork_factor"] = (resurfacing_agg["area"]/resurfacing_agg["total_area"]).round(4)

In [17]:
# Ensure roadwork_season is string
resurfacing_agg['roadwork_season'] = resurfacing_agg['roadwork_season'].astype(str)

# Create unique identifier
resurfacing_agg['normalized_street_name_season'] = (
    resurfacing_agg['normalized_street_name'] + "_summer_" + resurfacing_agg['roadwork_season']
)

# Optional: reorder columns so identifier is first
cols = ['normalized_street_name_season'] + [c for c in resurfacing_agg.columns if c not in ['normalized_street_name_season']]
resurfacing_agg = resurfacing_agg[cols]


In [18]:
resurfacing_agg["roadwork_season"] = resurfacing_agg["roadwork_season"].astype(int)

In [19]:
resurfacing_agg.drop_duplicates(inplace = True)

In [20]:
resurfacing_agg.head(20)

,normalized_street_name_season,normalized_street_name,roadwork_season,area,total_area,roadwork_factor
0,1-1 avenue_summer_2003,1-1 avenue,2003,375912.4,253446.9,1.4832
5,1-1 avenue_summer_2008,1-1 avenue,2008,105248.3,253446.9,0.4153
10,1-1 avenue_summer_2009,1-1 avenue,2009,24890.8,253446.9,0.0982
15,1-1 avenue_summer_2010,1-1 avenue,2010,395036.6,253446.9,1.5587
20,1-1 avenue_summer_2012,1-1 avenue,2012,12996.7,253446.9,0.0513
25,1-1 avenue_summer_2013,1-1 avenue,2013,656909.8,253446.9,2.5919
30,1-1 avenue_summer_2015,1-1 avenue,2015,55711.4,253446.9,0.2198
35,1-1 avenue_summer_2016,1-1 avenue,2016,22967.8,253446.9,0.0906
40,1-1 avenue_summer_2019,1-1 avenue,2019,25259.9,253446.9,0.0997
45,1-1 avenue_summer_2020,1-1 avenue,2020,252595.2,253446.9,0.9966


In [21]:
resurfacing_agg.to_csv("data/derived/resurfacing_agg_full.csv", index = False)